In [2]:
# ============================================================
# 03 - FEATURE ENGINEERING
# Healthcare 30-Day Readmission Prediction
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("=" * 70)
print("FEATURE ENGINEERING STARTED")
print("=" * 70)


# ============================================================
# 1. AUTOMATICALLY FIND DATASET
# ============================================================

data_dir = Path("../data")

csv_files = list(data_dir.rglob("*.csv"))

print("\nCSV files found:")

for file in csv_files:
    print(" -", file)


if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file found inside the data folder."
    )


# Find the dataset containing the readmitted column
selected_file = None

for file in csv_files:
    try:
        temp = pd.read_csv(file, nrows=5)

        if "readmitted" in temp.columns:
            selected_file = file
            break

        if "readmitted_30" in temp.columns:
            selected_file = file
            break

    except Exception:
        continue


if selected_file is None:

    if len(csv_files) == 1:
        selected_file = csv_files[0]

    else:
        raise ValueError(
            "Multiple CSV files found but none contains "
            "'readmitted' or 'readmitted_30'."
        )


print("\nSelected dataset:")
print(selected_file)


# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(selected_file)

print("\nDataset shape:")
print(df.shape)


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()


# ============================================================
# 4. CREATE TARGET
# ============================================================

if "readmitted" in df.columns:

    df["readmitted_30"] = (
        df["readmitted"]
        .astype(str)
        .str.strip()
        .eq("<30")
        .astype(int)
    )

elif "readmitted_30" in df.columns:

    df["readmitted_30"] = (
        pd.to_numeric(
            df["readmitted_30"],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )

else:

    raise ValueError(
        "Could not find readmission target."
    )


print("\nTarget distribution:")
print(
    df["readmitted_30"]
    .value_counts()
    .sort_index()
)

print("\nTarget percentage:")
print(
    (
        df["readmitted_30"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)


# ============================================================
# 5. REMOVE ID / TARGET LEAKAGE
# ============================================================

drop_cols = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "readmitted_30"
]

drop_cols = [
    col for col in drop_cols
    if col in df.columns
]

X = df.drop(columns=drop_cols)
y = df["readmitted_30"]


print("\nRemoved columns:")
for col in drop_cols:
    print(" -", col)


# ============================================================
# 6. REPLACE UNKNOWN VALUES
# ============================================================

X = X.replace("?", np.nan)


# ============================================================
# 7. AGE FEATURE ENGINEERING
# ============================================================

if "age" in X.columns:

    age_mapping = {
        "[0-10)": 5,
        "[10-20)": 15,
        "[20-30)": 25,
        "[30-40)": 35,
        "[40-50)": 45,
        "[50-60)": 55,
        "[60-70)": 65,
        "[70-80)": 75,
        "[80-90)": 85,
        "[90-100)": 95
    }

    X["age_numeric"] = X["age"].map(age_mapping)

    X.drop(
        columns=["age"],
        inplace=True
    )

    print("\nAge converted to numeric midpoint.")


# ============================================================
# 8. REMOVE EXTREMELY HIGH-CARDINALITY IDENTIFIERS
# ============================================================

# Some hospital dataset columns can behave like identifiers.
# We remove columns with almost unique values.

high_cardinality_cols = []

for col in X.columns:

    unique_ratio = X[col].nunique(
        dropna=False
    ) / len(X)

    if unique_ratio > 0.95:
        high_cardinality_cols.append(col)


# Don't remove useful numeric variables automatically
# unless they are obvious IDs.

for col in high_cardinality_cols:

    if (
        "id" in col.lower()
        or "number" in col.lower()
        or "patient" in col.lower()
    ):

        X.drop(
            columns=[col],
            inplace=True
        )

        print(
            "Removed high-cardinality identifier:",
            col
        )


# ============================================================
# 9. IDENTIFY NUMERIC / CATEGORICAL FEATURES
# ============================================================

numeric_features = X.select_dtypes(
    include=[
        "int64",
        "float64",
        "int32",
        "float32"
    ]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns.tolist()


print("\nFeature types:")
print(
    "Numeric features    :",
    len(numeric_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)


# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)


print("\nTrain/Test split:")
print(
    "X_train:",
    X_train.shape
)

print(
    "X_test :",
    X_test.shape
)


# ============================================================
# 11. NUMERICAL PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# 12. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# ============================================================
# 13. COMBINE PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],

    remainder="drop"
)


# ============================================================
# 14. FIT ON TRAINING DATA ONLY
# ============================================================

X_train_processed = (
    preprocessor
    .fit_transform(X_train)
)

X_test_processed = (
    preprocessor
    .transform(X_test)
)


# ============================================================
# 15. FEATURE NAMES
# ============================================================

feature_names = (
    preprocessor
    .get_feature_names_out()
)


X_train_processed = pd.DataFrame(

    X_train_processed,

    columns=feature_names,

    index=X_train.index
)


X_test_processed = pd.DataFrame(

    X_test_processed,

    columns=feature_names,

    index=X_test.index
)


# ============================================================
# 16. FINAL VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("FEATURE ENGINEERING COMPLETED")
print("=" * 70)

print(
    "\nOriginal feature count:",
    X.shape[1]
)

print(
    "Final feature count   :",
    X_train_processed.shape[1]
)

print(
    "\nX_train processed:",
    X_train_processed.shape
)

print(
    "X_test processed :",
    X_test_processed.shape
)


print("\nTarget distribution - TRAIN:")

print(
    y_train.value_counts()
)


print("\nTarget distribution - TEST:")

print(
    y_test.value_counts()
)


print(
    "\nRemaining NaN in X_train:",
    X_train_processed
    .isna()
    .sum()
    .sum()
)


print(
    "Remaining NaN in X_test :",
    X_test_processed
    .isna()
    .sum()
    .sum()
)


print("\nSample processed data:")

display(
    X_train_processed.head()
)


print("\n")
print("=" * 70)
print("READY FOR MODEL BUILDING")
print("=" * 70)

FEATURE ENGINEERING STARTED

CSV files found:
 - ../data/cleaned_healthcare_data.csv

Selected dataset:
../data/cleaned_healthcare_data.csv

Dataset shape:
(101766, 46)

Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64

Target percentage:
readmitted_30
0    88.84
1    11.16
Name: proportion, dtype: float64

Removed columns:
 - readmitted_30

Age converted to numeric midpoint.


/var/folders/fv/j7501lm50cvcgmdc4ncsgpp00000gn/T/ipykernel_15755/1115482842.py:258: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(



Feature types:
Numeric features    : 12
Categorical features: 33

Train/Test split:
X_train: (81412, 45)
X_test : (20354, 45)


FEATURE ENGINEERING COMPLETED

Original feature count: 45
Final feature count   : 2356

X_train processed: (81412, 2356)
X_test processed : (20354, 2356)

Target distribution - TRAIN:
readmitted_30
0    72326
1     9086
Name: count, dtype: int64

Target distribution - TEST:
readmitted_30
0    18083
1     2271
Name: count, dtype: int64

Remaining NaN in X_train: 0
Remaining NaN in X_test : 0

Sample processed data:


,numeric__admission_type_id,numeric__discharge_disposition_id,numeric__admission_source_id,numeric__time_in_hospital,numeric__num_lab_procedures,numeric__num_procedures,numeric__num_medications,numeric__number_outpatient,numeric__number_emergency,numeric__number_inpatient,...,categorical__glimepiride-pioglitazone_No,categorical__glimepiride-pioglitazone_Steady,categorical__metformin-rosiglitazone_No,categorical__metformin-rosiglitazone_Steady,categorical__metformin-pioglitazone_No,categorical__metformin-pioglitazone_Steady,categorical__change_Ch,categorical__change_No,categorical__diabetesMed_No,categorical__diabetesMed_Yes
46943,-0.706932,3.645771,-0.187628,2.211912,0.096863,-0.198173,0.243726,-0.291734,-0.207009,-0.50194,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
60886,-0.706932,-0.514411,0.305517,-1.135963,-0.563264,1.563785,-0.125745,1.285124,-0.207009,-0.50194,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
34417,-0.014635,-0.514411,-1.173919,0.203187,0.147642,0.389146,-0.002588,-0.291734,-0.207009,-0.50194,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
57371,-0.706932,-0.514411,0.305517,-1.135963,-0.360148,-0.785492,-0.618374,-0.291734,-0.207009,-0.50194,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
19626,-0.706932,-0.136213,-0.187628,1.877125,0.553874,0.976466,-0.002588,-0.291734,-0.207009,-0.50194,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0




READY FOR MODEL BUILDING
